# DineIQ — Exploratory Data Analysis (SRS Step 8)

Reads the latest feature snapshot from `parquet_data/features/` (built by
`spark_jobs/04_feature_engineering.py`) plus the order-line fact table for the time,
channel and promotion breakdowns. Writes charts to `reports/figures/` and the findings to
`reports/eda_report.md`.

Revenue and volume exclude cancelled orders throughout. Peak hours are 12:00-14:00 and
19:00-22:00; the weekend is Friday-Sunday.

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import pyspark.sql.functions as F

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "spark_jobs"))
from schemas import load_clean_table
from spark_utils import PARQUET_DIR, PROCESSED_DIR, get_spark

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
REPORT = ROOT / "reports" / "eda_report.md"
TOP_N = 15

spark = get_spark("dineiq-eda")
spark.sparkContext.setLogLevel("ERROR")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/24 22:33:29 WARN Utils: Your hostname, Muzammil, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/24 22:33:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/09/24 22:33:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
BLUE, INK, INK2, GRID = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
})


def hbar(df, label, value, title, fname, xlabel, fmt=None):
    d = df.iloc[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 0.32 * len(d) + 1.1))
    ax.barh(d[label], d[value], color=BLUE, height=0.7)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    if fmt:
        ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(fmt))
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=130)
    plt.close(fig)
    return f"figures/{fname}"


def millions(x, _=None):
    return f"{x / 1e6:,.0f}M"


def md_table(df, formats=None):
    formats = formats or {}
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if pd.isna(v):
                cells.append("—")
            elif c in formats:
                cells.append(formats[c](v))
            else:
                cells.append(str(v))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


pkr = lambda v: f"{v:,.0f}"
pct = lambda v: f"{v:.1f}%"
frac_pct = lambda v: f"{v * 100:.1f}%"
num2 = lambda v: f"{v:.2f}"
intf = lambda v: f"{int(v):,}"

## Load the latest feature snapshot and the fact table

In [3]:
snapshots = sorted(p.name.split("=")[1] for p in (PARQUET_DIR / "features" / "item_features").glob("as_of_date=*"))
AS_OF = snapshots[-1]
print("available snapshots:", snapshots, "-> using", AS_OF)

def features(name):
    return pd.read_parquet(PARQUET_DIR / "features" / name / f"as_of_date={AS_OF}")

items = features("item_features")
customers = features("customer_features")
locations = features("location_features")

menu = load_clean_table(spark, "Menu_Items", PROCESSED_DIR).select(
    "item_id", "launch_date", "is_active", "is_seasonal", "season").toPandas()
items = items.merge(menu, on="item_id", how="left")

ratings = load_clean_table(spark, "Ratings", PROCESSED_DIR).filter(F.col("rating_date") <= F.lit(AS_OF))
rating_counts = (ratings.filter(F.col("item_id").isNotNull() & F.col("rating_value").isNotNull())
                 .groupBy("item_id").count().withColumnRenamed("count", "n_ratings").toPandas())
items = items.merge(rating_counts, on="item_id", how="left")

cutoff = F.lit(pd.Timestamp(AS_OF) + pd.Timedelta(days=1)).cast("timestamp")
fact_all = spark.read.parquet(str(PARQUET_DIR / "fact_order_line")).filter(F.col("order_datetime") < cutoff)
fact = (fact_all.filter(F.col("order_status") != "Cancelled")
        .withColumn("line_total", F.col("line_total").cast("double"))
        .withColumn("line_cost", F.col("quantity") * F.col("unit_cost").cast("double"))
        .cache())
orders_all = spark.read.parquet(str(PARQUET_DIR / "orders")).filter(F.col("order_datetime") < cutoff)
orders = orders_all.filter(F.col("order_status") != "Cancelled").cache()

TOTAL_REVENUE = items["item_revenue"].sum()
TOTAL_MARGIN = items["contribution_margin"].sum()
N_ORDERS = orders.count()
print(f"{len(items)} items, {len(customers):,} customers, {len(locations)} locations, {N_ORDERS:,} orders")
print(f"revenue {TOTAL_REVENUE:,.0f} PKR, gross margin {TOTAL_MARGIN / TOTAL_REVENUE:.1%}")

available snapshots: ['2025-09-30', '2025-12-31'] -> using 2025-12-31


/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


160 items, 49,606 customers, 25 locations, 179,324 orders
revenue 1,200,430,365 PKR, gross margin 56.5%


## Dish rankings

In [4]:
R = {}
FIGS = {}
base_cols = ["item_name", "category_name"]

top_sell = items.nlargest(TOP_N, "total_quantity_sold")
R["top_selling"] = top_sell[base_cols + ["total_quantity_sold", "item_revenue", "profit_percentage", "average_rating"]]
FIGS["top_selling"] = hbar(top_sell, "item_name", "total_quantity_sold",
                           f"Top {TOP_N} dishes by units sold", "top_selling_dishes.png", "Units sold",
                           lambda x, _: f"{x:,.0f}")

# Items launched during the year have had fewer days to sell, so show a per-day rate too.
year_start = pd.Timestamp(AS_OF).replace(month=1, day=1)
first_day = pd.to_datetime(items["launch_date"]).clip(lower=year_start)
items["days_on_menu"] = (pd.Timestamp(AS_OF) - first_day).dt.days + 1
items["units_per_day"] = items["total_quantity_sold"] / items["days_on_menu"]
low_sell = items.nsmallest(TOP_N, "total_quantity_sold")
R["lowest_selling"] = low_sell[base_cols + ["total_quantity_sold", "units_per_day", "launch_date", "item_revenue"]]
established = items[items["days_on_menu"] >= 180]
low_rate = established.nsmallest(TOP_N, "units_per_day")
R["lowest_rate"] = low_rate[base_cols + ["units_per_day", "total_quantity_sold", "profit_percentage", "average_rating"]]
NEW_IN_BOTTOM = int((low_sell["days_on_menu"] < 180).sum())

top_rev = items.nlargest(TOP_N, "item_revenue")
R["highest_revenue"] = top_rev[base_cols + ["item_revenue", "total_quantity_sold", "profit_percentage"]]
FIGS["highest_revenue"] = hbar(top_rev, "item_name", "item_revenue",
                               f"Top {TOP_N} dishes by revenue (PKR)", "highest_revenue_dishes.png", "Revenue (PKR)", millions)

top_profit = items.nlargest(TOP_N, "contribution_margin")
R["highest_profit"] = top_profit[base_cols + ["contribution_margin", "item_revenue", "profit_percentage", "total_quantity_sold"]]
FIGS["highest_profit"] = hbar(top_profit, "item_name", "contribution_margin",
                              f"Top {TOP_N} dishes by contribution margin (PKR)", "highest_profit_dishes.png",
                              "Contribution margin (PKR)", millions)

top_margin = items.nlargest(TOP_N, "profit_percentage")
R["highest_margin"] = top_margin[base_cols + ["profit_percentage", "contribution_margin", "total_quantity_sold", "item_popularity"]]
low_margin = items.nsmallest(TOP_N, "profit_percentage")
R["lowest_margin"] = low_margin[base_cols + ["profit_percentage", "total_quantity_sold", "item_popularity"]]

for k, v in R.items():
    print(k); display(v)

top_selling


,item_name,category_name,total_quantity_sold,item_revenue,profit_percentage,average_rating
159,Plain Naan,Sides & Breads,87655,5448821.98,30.359029,3.619369
158,Plain Fries,Sides & Breads,87102,27790441.38,35.104522,3.670659
157,Soft Drink (Regular),Cold Beverages & Shakes,64861,10158561.40,32.290401,3.919181
156,Zinger Burger,Burgers & Wraps,45630,30673764.69,30.306312,3.536058
155,Mineral Water (500ml),Cold Beverages & Shakes,39854,4059409.21,32.063950,4.000000
154,Fresh Lime Soda,Cold Beverages & Shakes,38787,10068667.73,57.930284,3.680745
153,Chicken Karahi (Half),Karahi & Handi,38716,55987138.16,58.947404,4.100871
152,Chicken Tikka Pizza (Medium),Pizza,35362,51983381.16,29.522088,3.559739
151,Chicken Handi,Karahi & Handi,33386,46728104.87,58.683168,3.658523
150,Chicken Biryani,Rice & Biryani,31877,21306479.72,32.627619,3.637115


lowest_selling


,item_name,category_name,total_quantity_sold,units_per_day,launch_date,item_revenue
0,Pink Sauce Pasta,Pasta,728,9.972603,2025-10-20,828884.18
1,Kunafa Cheesecake,Desserts,844,27.225806,2025-12-01,749884.59
2,Korean Fried Chicken Burger,Burgers & Wraps,854,14.474576,2025-11-03,903865.79
3,Fish and Chips,Seafood,1084,2.969863,2020-12-15,1507318.57
4,Chicken Wings Bucket,Appetizers,1173,3.213699,2021-10-11,1675051.94
5,Nashville Hot Wrap,Burgers & Wraps,1239,32.605263,2025-11-24,1102254.64
6,Truffle Mushroom Pasta,Pasta,1266,3.468493,2022-09-30,2237934.32
7,Brownie Sundae,Desserts,1266,3.468493,2018-12-06,776098.58
8,Molten Lava Cake,Desserts,1512,4.142466,2023-07-08,1180505.41
9,Smash Burger,Burgers & Wraps,1576,18.114943,2025-10-06,1902531.57


lowest_rate


...

,item_name,category_name,contribution_margin,item_revenue,profit_percentage,total_quantity_sold
146,Mutton Karahi (Half),Karahi & Handi,3.407618e+07,58637744.40,58.113050,22479
153,Chicken Karahi (Half),Karahi & Handi,3.300296e+07,55987138.16,58.947404,38716
151,Chicken Handi,Karahi & Handi,2.742153e+07,46728104.87,58.683168,33386
82,Mutton Raan,BBQ & Grills,1.888148e+07,31813827.63,59.349903,6342
152,Chicken Tikka Pizza (Medium),Pizza,1.534658e+07,51983381.16,29.522088,35362
140,Nihari,Karahi & Handi,1.497460e+07,22290377.27,67.179665,17733
134,Peshawari Karahi,Karahi & Handi,1.467636e+07,26730276.67,54.905398,15954
149,Loaded Fries,Sides & Breads,1.420898e+07,24031549.19,59.126352,31314
115,Mutton Handi,Karahi & Handi,1.404730e+07,25544052.48,54.992437,10362
116,Kebab Crust Pizza (Medium),Pizza,1.276004e+07,20728404.26,61.558256,11152


highest_margin


,item_name,category_name,profit_percentage,contribution_margin,total_quantity_sold,item_popularity
25,Cafe Latte,Hot Beverages,85.819035,1056854.66,2489,0.157233
52,Blue Lagoon,Cold Beverages & Shakes,85.071138,1557073.50,4131,0.327044
35,Tiramisu,Desserts,84.446497,2515146.75,3288,0.220126
32,Cappuccino,Hot Beverages,84.230684,1121219.67,2759,0.201258
17,Loaded Nachos,Appetizers,83.865100,1674570.20,2022,0.106918
6,Truffle Mushroom Pasta,Pasta,83.451903,1867598.78,1266,0.037736
97,Mint Margarita,Cold Beverages & Shakes,82.745760,2737919.67,8455,0.610063
8,Molten Lava Cake,Desserts,82.321287,971807.25,1512,0.050314
47,Kulfi,Desserts,69.042642,746281.51,3934,0.295597
55,Falooda,Desserts,68.919552,1749898.02,4243,0.345912


lowest_margin


,item_name,category_name,profit_percentage,total_quantity_sold,item_popularity
143,Doodh Patti Chai,Hot Beverages,28.887889,21390,0.899371
152,Chicken Tikka Pizza (Medium),Pizza,29.522088,35362,0.955975
156,Zinger Burger,Burgers & Wraps,30.306312,45630,0.981132
159,Plain Naan,Sides & Breads,30.359029,87655,1.000000
155,Mineral Water (500ml),Cold Beverages & Shakes,32.063950,39854,0.974843
157,Soft Drink (Regular),Cold Beverages & Shakes,32.290401,64861,0.987421
150,Chicken Biryani,Rice & Biryani,32.627619,31877,0.943396
158,Plain Fries,Sides & Breads,35.104522,87102,0.993711
39,Jumbo Family Pizza,Pizza,43.421118,3545,0.245283
7,Brownie Sundae,Desserts,45.829032,1266,0.037736


In [5]:
waste = items.nlargest(TOP_N, "wastage_percentage")
R["high_wastage"] = waste[base_cols + ["wastage_percentage", "wastage_cost", "total_quantity_sold", "promotion_dependency"]]
FIGS["high_wastage"] = hbar(waste, "item_name", "wastage_percentage",
                            f"Top {TOP_N} dishes by wastage rate", "high_wastage_dishes.png",
                            "Wasted portions / (wasted + sold)", lambda x, _: f"{x:.0%}")
waste_cost = items.nlargest(TOP_N, "wastage_cost")
R["high_wastage_cost"] = waste_cost[base_cols + ["wastage_cost", "wastage_percentage", "total_quantity_sold"]]
WASTE_TOTAL = items["wastage_cost"].sum()
WASTE_TOP6_SHARE = items.nlargest(6, "wastage_cost")["wastage_cost"].sum() / WASTE_TOTAL

MIN_RATINGS = 30
rated = items[items["n_ratings"] >= MIN_RATINGS]
best = rated.nlargest(TOP_N, "average_rating")
worst = rated.nsmallest(TOP_N, "average_rating")
R["best_rated"] = best[base_cols + ["average_rating", "n_ratings", "rating_trend", "total_quantity_sold"]]
R["worst_rated"] = worst[base_cols + ["average_rating", "n_ratings", "rating_trend", "total_quantity_sold", "repeat_purchase_rate"]]
FIGS["worst_rated"] = hbar(worst, "item_name", "average_rating",
                           f"Lowest-rated {TOP_N} dishes (avg rating, 1-5)", "poorly_rated_dishes.png", "Average rating")
AVG_RATING = (items["average_rating"] * items["n_ratings"]).sum() / items["n_ratings"].sum()
print(f"total wastage cost {WASTE_TOTAL:,.0f}; top-6 share {WASTE_TOP6_SHARE:.1%}; overall avg rating {AVG_RATING:.2f}")
display(R["high_wastage"]); display(R["high_wastage_cost"]); display(R["best_rated"]); display(R["worst_rated"])

total wastage cost 28,521,532; top-6 share 69.3%; overall avg rating 3.70


,item_name,category_name,wastage_percentage,wastage_cost,total_quantity_sold,promotion_dependency
4,Chicken Wings Bucket,Appetizers,0.387146,525877.60,1173,0.746706
99,Fresh Garden Salad,Salads,0.363661,1013430.78,8905,0.181304
7,Brownie Sundae,Desserts,0.336255,212986.12,1266,0.562981
151,Chicken Handi,Karahi & Handi,0.196890,4733201.30,33386,0.190574
146,Mutton Karahi (Half),Karahi & Handi,0.195657,5974610.12,22479,0.198653
144,Seekh Kabab,BBQ & Grills,0.195273,1573843.28,21655,0.191135
6,Truffle Mushroom Pasta,Pasta,0.191991,87995.57,1266,0.147857
153,Chicken Karahi (Half),Karahi & Handi,0.189583,5376754.09,38716,0.189103
92,Fried Fish Lahori,Seafood,0.188461,1084532.77,7623,0.157603
34,Double Patty Burger,Burgers & Wraps,0.168827,415278.96,3269,0.665476


,item_name,category_name,wastage_cost,wastage_percentage,total_quantity_sold
146,Mutton Karahi (Half),Karahi & Handi,5974610.12,0.195657,22479
153,Chicken Karahi (Half),Karahi & Handi,5376754.09,0.189583,38716
151,Chicken Handi,Karahi & Handi,4733201.30,0.196890,33386
144,Seekh Kabab,BBQ & Grills,1573843.28,0.195273,21655
92,Fried Fish Lahori,Seafood,1084532.77,0.188461,7623
99,Fresh Garden Salad,Salads,1013430.78,0.363661,8905
39,Jumbo Family Pizza,Pizza,936845.69,0.154522,3545
4,Chicken Wings Bucket,Appetizers,525877.60,0.387146,1173
34,Double Patty Burger,Burgers & Wraps,415278.96,0.168827,3269
152,Chicken Tikka Pizza (Medium),Pizza,321737.08,0.008705,35362


,item_name,category_name,average_rating,n_ratings,rating_trend,total_quantity_sold
67,Margherita Pizza (Medium),Pizza,4.274247,299,0.005576,5019
41,Chicken Wings (6 pcs),Appetizers,4.236842,228,-0.019551,3591
122,Hot Chocolate,Hot Beverages,4.218182,550,0.002032,12529
6,Truffle Mushroom Pasta,Pasta,4.197183,71,-0.006585,1266
48,Beef Bihari Boti,BBQ & Grills,4.185606,264,0.008808,3997
20,Prawn Karahi,Seafood,4.164179,134,-0.006612,2200
25,Cafe Latte,Hot Beverages,4.160714,112,0.004100,2489
35,Tiramisu,Desserts,4.156425,179,0.012148,3288
71,Pepperoni Pizza (Medium),Pizza,4.119403,335,0.002345,5396
23,Grilled Prawns Platter,Seafood,4.118110,127,-0.027973,2393


,item_name,category_name,average_rating,n_ratings,rating_trend,total_quantity_sold,repeat_purchase_rate
61,Veg Spring Rolls,Appetizers,1.868421,418,-0.010794,4689,0.090234
49,Beef Lasagna,Pasta,1.890585,393,0.001079,4052,0.091346
70,Fish Burger,Burgers & Wraps,2.006263,479,-0.010690,5205,0.100757
103,Cold Coffee,Cold Beverages & Shakes,2.055743,592,-0.000332,8973,0.120899
118,Mutton Pulao,Rice & Biryani,2.212245,980,-0.005812,11460,0.156387
104,Fajita Pizza (Medium),Pizza,3.264286,560,0.005199,8979,0.147123
36,Arrabbiata Pasta,Pasta,3.320197,203,-0.000150,3351,0.080960
57,Chocolate Fudge Cake,Desserts,3.322835,254,0.004421,4299,0.103734
7,Brownie Sundae,Desserts,3.323529,68,-0.032349,1266,0.017076
144,Seekh Kabab,BBQ & Grills,3.337428,1221,-0.007289,21655,0.221677


In [6]:
# Volume vs margin: the high-volume dishes are not the high-margin ones.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(items["total_quantity_sold"], items["profit_percentage"], s=22, color=BLUE, alpha=0.75,
           edgecolor="#fcfcfb", linewidth=0.8)
labelled = [(items.nlargest(1, "total_quantity_sold").iloc[0], (-8, -12), "right"),
            (items.nlargest(1, "profit_percentage").iloc[0], (6, 0), "left"),
            (items.nsmallest(1, "profit_percentage").iloc[0], (6, -3), "left")]
for r, offset, ha in labelled:
    ax.annotate(r["item_name"], (r["total_quantity_sold"], r["profit_percentage"]), fontsize=8,
                color=INK2, xytext=offset, textcoords="offset points", ha=ha, va="center")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_xlabel("Units sold (log scale)")
ax.set_ylabel("Profit margin (%)")
ax.set_title("Units sold vs profit margin, per dish")
ax.grid(color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIG_DIR / "volume_vs_margin.png", dpi=130)
plt.close(fig)
FIGS["volume_vs_margin"] = "figures/volume_vs_margin.png"
VOL_MARGIN_CORR = items[["total_quantity_sold", "profit_percentage"]].corr(method="spearman").iloc[0, 1]
MEDIAN_MARGIN = items["profit_percentage"].median()
TOP10_VOL_MARGIN = items.nlargest(10, "total_quantity_sold")["profit_percentage"].mean()
print(f"spearman(volume, margin) = {VOL_MARGIN_CORR:.2f}; median margin {MEDIAN_MARGIN:.1f}%; top-10-volume avg margin {TOP10_VOL_MARGIN:.1f}%")

spearman(volume, margin) = -0.07; median margin 59.4%; top-10-volume avg margin 39.8%


## Menu categories

In [7]:
cat = (fact.groupBy("category_name").agg(
            F.sum("line_total").alias("revenue"),
            F.sum("line_cost").alias("cost"),
            F.countDistinct("order_id").alias("orders"),
            F.sum("quantity").alias("units"))
       .toPandas())
cat["margin_pct"] = (cat["revenue"] - cat["cost"]) / cat["revenue"] * 100
cat["revenue_share"] = cat["revenue"] / cat["revenue"].sum()
cat["order_penetration"] = cat["orders"] / N_ORDERS
cat_waste = items.groupby("category_name")["wastage_cost"].sum().rename("wastage_cost")
cat = cat.merge(cat_waste, on="category_name", how="left")
cat_rev = cat.sort_values("revenue", ascending=False)
cat_ord = cat.sort_values("orders", ascending=False)
R["category_revenue"] = cat_rev[["category_name", "revenue", "revenue_share", "margin_pct", "units", "wastage_cost"]]
R["category_orders"] = cat_ord[["category_name", "orders", "order_penetration", "units"]]
FIGS["category_revenue"] = hbar(cat_rev, "category_name", "revenue", "Revenue by menu category (PKR)",
                                "category_revenue.png", "Revenue (PKR)", millions)
FIGS["category_orders"] = hbar(cat_ord, "category_name", "order_penetration",
                               "Share of orders containing each category", "category_orders.png",
                               "Share of orders", lambda x, _: f"{x:.0%}")
display(R["category_revenue"]); display(R["category_orders"])

/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,category_name,revenue,revenue_share,margin_pct,units,wastage_cost
1,Karahi & Handi,2.809401e+08,0.234033,59.177835,185889,16347107.90
6,Pizza,1.602933e+08,0.133530,49.546235,98158,2449859.70
2,BBQ & Grills,1.574535e+08,0.131164,58.648197,109635,2026879.22
7,Rice & Biryani,9.967938e+07,0.083036,55.812700,125096,193734.93
10,Sides & Breads,8.419736e+07,0.070139,49.367588,339277,398661.93
13,Cold Beverages & Shakes,8.089221e+07,0.067386,58.754843,268767,162792.23
12,Burgers & Wraps,7.956728e+07,0.066282,46.857794,104607,1265279.26
11,Seafood,5.723646e+07,0.047680,60.629075,30876,1205220.79
9,Appetizers,4.858830e+07,0.040476,61.779523,88890,1141306.02
0,Pasta,4.454414e+07,0.037107,60.996654,34982,1147726.21


,category_name,orders,order_penetration,units
10,Sides & Breads,101704,0.567152,339277
13,Cold Beverages & Shakes,98416,0.548817,268767
1,Karahi & Handi,91362,0.509480,185889
7,Rice & Biryani,70408,0.392630,125096
2,BBQ & Grills,63028,0.351476,109635
12,Burgers & Wraps,60746,0.338750,104607
6,Pizza,57106,0.318452,98158
9,Appetizers,51764,0.288662,88890
5,Hot Beverages,45105,0.251528,95963
8,Desserts,42957,0.239550,69042


## Peak ordering periods

In [8]:
DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
o_time = (orders.select(F.hour("order_datetime").alias("hour"),
                        ((F.dayofweek("order_datetime") + 5) % 7).alias("dow"),
                        F.to_date("order_datetime").alias("day"),
                        F.month("order_datetime").alias("month"))
          .toPandas())
heat = o_time.groupby(["dow", "hour"]).size().unstack(fill_value=0).reindex(index=range(7), columns=range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(9, 3.4))
im = ax.imshow(heat.values, aspect="auto", cmap="Blues")
ax.set_yticks(range(7), DOW)
ax.set_xticks(range(24), [f"{h:02d}" for h in range(24)])
ax.set_xlabel("Hour of day")
ax.set_title("Orders by day of week and hour")
for s in ax.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
cb.set_label("Orders", color=INK2)
cb.outline.set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "peak_hours_heatmap.png", dpi=130)
plt.close(fig)
FIGS["heatmap"] = "figures/peak_hours_heatmap.png"

hour_share = o_time.groupby("hour").size() / len(o_time)
PEAK_SHARE = hour_share.loc[[12, 13, 19, 20, 21]].sum()
TOP_HOURS = hour_share.sort_values(ascending=False).head(5)

days_per_dow = o_time.groupby("dow")["day"].nunique()
per_day = o_time.groupby("dow").size() / days_per_dow
dow_tbl = pd.DataFrame({"day": DOW, "avg_orders_per_day": per_day.values,
                        "share_of_week": (per_day / per_day.sum()).values})
WEEKEND_LIFT = per_day.loc[[4, 5, 6]].mean() / per_day.loc[[0, 1, 2, 3]].mean() - 1
R["dow"] = dow_tbl

ramadan = o_time[o_time["month"] == 3].groupby("hour").size() / (o_time["month"] == 3).sum()
other = o_time[o_time["month"] != 3].groupby("hour").size() / (o_time["month"] != 3).sum()
RAMADAN_PEAK_HOUR = int(ramadan.idxmax())
RAMADAN_LUNCH = ramadan.reindex([12, 13]).fillna(0).sum()
OTHER_LUNCH = other.reindex([12, 13]).fillna(0).sum()

hourly = o_time.groupby(["month", "hour"]).size()
print(f"peak-window share {PEAK_SHARE:.1%}; weekend lift {WEEKEND_LIFT:.1%}")
print(TOP_HOURS)
print(f"Ramadan (March) busiest hour {RAMADAN_PEAK_HOUR}:00, lunch share {RAMADAN_LUNCH:.1%} vs {OTHER_LUNCH:.1%} other months")
display(dow_tbl)

/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


peak-window share 55.9%; weekend lift 43.7%
hour
20    0.127997
21    0.117547
13    0.113956
19    0.105457
12    0.093780
dtype: float64
Ramadan (March) busiest hour 18:00, lunch share 2.8% vs 22.2% other months


,day,avg_orders_per_day,share_of_week
0,Mon,403.346154,0.117231
1,Tue,397.942308,0.115661
2,Wed,412.509434,0.119894
3,Thu,442.423077,0.128589
4,Fri,579.596154,0.168458
5,Sat,624.230769,0.181430
6,Sun,580.557692,0.168737


## Location-wise sales patterns

In [9]:
loc = locations.sort_values("location_revenue", ascending=False).copy()
loc["profit_margin_pct"] = loc["location_profit"] / loc["location_revenue"] * 100
R["locations"] = loc[["location_name", "city", "location_type", "location_revenue", "profit_margin_pct",
                      "location_avg_order_value", "location_customer_count",
                      "location_repeat_customer_rate", "location_wastage_rate"]]
loc_plot = loc.assign(label=loc["location_name"].str.replace("DineIQ ", "", regex=False))
FIGS["locations"] = hbar(loc_plot, "label", "location_revenue", "Revenue by location (PKR)",
                         "location_revenue.png", "Revenue (PKR)", millions)
LOC_SPREAD = loc["location_revenue"].max() / loc["location_revenue"].min()

lt = (orders.select("location_id", "order_datetime")
      .join(load_clean_table(spark, "Restaurants", PROCESSED_DIR).select("location_id", "location_type"), "location_id")
      .groupBy("location_type").agg(
          F.count("*").alias("orders"),
          F.avg(F.dayofweek("order_datetime").isin(6, 7, 1).cast("double")).alias("weekend_share"),
          F.avg(((F.hour("order_datetime") >= 12) & (F.hour("order_datetime") < 14)).cast("double")).alias("lunch_share"),
          F.avg(((F.hour("order_datetime") >= 19) & (F.hour("order_datetime") < 22)).cast("double")).alias("dinner_share"))
      .toPandas())
by_type = locations.groupby("location_type").agg(
    n_locations=("location_id", "count"), revenue=("location_revenue", "sum"),
    avg_order_value=("location_avg_order_value", "mean"),
    wastage_rate=("location_wastage_rate", "mean"),
    repeat_rate=("location_repeat_customer_rate", "mean")).reset_index()
by_type = by_type.merge(lt, on="location_type")
by_type["revenue_per_location"] = by_type["revenue"] / by_type["n_locations"]
by_type = by_type.sort_values("revenue_per_location", ascending=False)
R["location_types"] = by_type[["location_type", "n_locations", "revenue_per_location", "avg_order_value",
                               "weekend_share", "lunch_share", "dinner_share", "repeat_rate", "wastage_rate"]]
display(R["locations"]); display(R["location_types"])

/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,location_name,city,location_type,location_revenue,profit_margin_pct,location_avg_order_value,location_customer_count,location_repeat_customer_rate,location_wastage_rate
18,DineIQ Karachi Dolmen Mall Clifton,Karachi,Mall,78497202.56,57.204292,6727.562784,5717,0.262725,0.019825
14,DineIQ Faisalabad Lyallpur Galleria,Faisalabad,Mall,77683747.04,57.315719,7366.879757,5296,0.266994,0.024503
1,DineIQ Quetta Jinnah Road,Quetta,Downtown,70423224.91,56.285449,6744.873567,5049,0.263418,0.018814
16,DineIQ Karachi Saddar,Karachi,Downtown,70069749.05,56.439579,6562.066777,5258,0.259224,0.017607
2,DineIQ Lahore MM Alam Road,Lahore,Downtown,58433822.98,55.334260,6373.671791,4555,0.252689,0.024925
7,DineIQ Karachi Gulshan-e-Iqbal,Karachi,Suburban,55079496.75,56.387864,6525.233592,4111,0.249574,0.020681
3,DineIQ Rawalpindi Bahria Town Phase 7,Rawalpindi,Suburban,54704621.48,55.206270,6257.678046,4347,0.256499,0.022005
9,DineIQ Islamabad F-7 Markaz,Islamabad,Urban,51253423.11,56.653123,6619.323661,4020,0.235075,0.026623
0,DineIQ Karachi Clifton Block 5,Karachi,Urban,50070146.40,55.799133,6725.338670,3779,0.248743,0.020269
20,DineIQ Lahore Gulberg III,Lahore,Urban,49968614.63,56.281292,6625.379824,3955,0.237168,0.025737


,location_type,n_locations,revenue_per_location,avg_order_value,weekend_share,lunch_share,dinner_share,repeat_rate,wastage_rate
2,Mall,4,6.230827e+07,6961.127842,0.564220,0.193164,0.362190,0.251552,0.023093
0,Downtown,5,5.718342e+07,6488.382590,0.440463,0.253138,0.324236,0.250033,0.021805
4,Urban,7,4.690027e+07,6672.184600,0.525121,0.194445,0.364531,0.239582,0.024549
3,Suburban,6,4.200928e+07,6481.523944,0.554355,0.197506,0.362496,0.238505,0.024396
1,Highway,3,2.830752e+07,7352.857626,0.507807,0.171322,0.321652,0.218135,0.032992


## Channel-wise ordering patterns

In [10]:
ch_rev = (fact.groupBy("order_channel").agg(F.sum("line_total").alias("revenue"),
                                            F.sum("line_cost").alias("cost"))
          .toPandas())
ch_orders = (orders_all.groupBy("order_channel").agg(
                 F.count("*").alias("all_orders"),
                 F.avg((F.col("order_status") == "Cancelled").cast("double")).alias("cancel_rate"))
             .toPandas())
baskets = fact.select("order_id", "basket_size").distinct()
ch_basket = (orders.join(baskets, "order_id", "left").groupBy("order_channel").agg(
                 F.count("*").alias("orders"),
                 F.avg("basket_size").alias("avg_basket_size"),
                 F.avg(F.col("promotion_id").isNotNull().cast("double")).alias("promo_order_share"),
                 F.avg(F.hour("order_datetime").isin(12, 13, 19, 20, 21).cast("double")).alias("peak_share"))
             .toPandas())
ch = ch_rev.merge(ch_orders, on="order_channel").merge(ch_basket, on="order_channel")
ch["revenue_share"] = ch["revenue"] / ch["revenue"].sum()
ch["avg_order_value"] = ch["revenue"] / ch["orders"]
ch["margin_pct"] = (ch["revenue"] - ch["cost"]) / ch["revenue"] * 100
ch = ch.sort_values("revenue", ascending=False)
R["channels"] = ch[["order_channel", "orders", "revenue", "revenue_share", "avg_order_value", "avg_basket_size",
                    "margin_pct", "cancel_rate", "promo_order_share", "peak_share"]]
FIGS["channels"] = hbar(ch, "order_channel", "revenue_share", "Revenue share by order channel",
                        "channel_revenue_share.png", "Share of revenue", lambda x, _: f"{x:.0%}")
pref = customers["channel_preference"].value_counts(normalize=True).rename("share_of_customers").reset_index()
R["channel_pref"] = pref
display(R["channels"]); display(pref)

/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,order_channel,orders,revenue,revenue_share,avg_order_value,avg_basket_size,margin_pct,cancel_rate,promo_order_share,peak_share
4,Dine-in,71622,5.933332e+08,0.494267,8284.231055,6.065555,56.555489,0.019374,0.192064,0.557315
2,App,36810,2.168792e+08,0.180668,5891.854141,5.103636,56.327065,0.059962,0.173513,0.562510
3,Takeaway,31433,1.717077e+08,0.143038,5462.656179,4.757838,56.445030,0.029546,0.166990,0.557567
0,ThirdPartyDelivery,26967,1.454951e+08,0.121202,5395.300851,4.739882,56.345427,0.081005,0.163904,0.560797
1,Website,12492,7.301527e+07,0.060824,5844.962182,5.050288,56.287500,0.060681,0.173631,0.554275


,channel_preference,share_of_customers
0,Dine-in,0.414950
1,App,0.234085
2,Takeaway,0.153590
3,ThirdPartyDelivery,0.136899
4,Website,0.060477


## Promotion-driven sales

In [11]:
fact_m = fact.withColumn("is_promo", F.col("promotion_id").isNotNull())
monthly = (fact_m.groupBy("order_month").agg(
               F.sum("line_total").alias("revenue"),
               F.sum("line_cost").alias("cost"),
               F.sum(F.when(F.col("is_promo"), F.col("line_total")).otherwise(0.0)).alias("promo_revenue"),
               F.countDistinct("order_id").alias("orders"),
               F.countDistinct("customer_id").alias("customers"))
           .toPandas().sort_values("order_month"))
wastage = spark.read.parquet(str(PARQUET_DIR / "wastage")).filter(F.col("date") <= F.lit(AS_OF))
w_month = (wastage.groupBy(F.month("date").alias("order_month"))
           .agg(F.sum(F.col("cost_of_waste").cast("double")).alias("wastage_cost")).toPandas())
monthly = monthly.merge(w_month, on="order_month", how="left")
monthly["margin_pct"] = (monthly["revenue"] - monthly["cost"]) / monthly["revenue"] * 100
monthly["promo_revenue_share"] = monthly["promo_revenue"] / monthly["revenue"]
monthly["month"] = pd.to_datetime(monthly["order_month"], format="%m").dt.strftime("%b")
R["monthly"] = monthly[["month", "orders", "customers", "revenue", "promo_revenue_share", "margin_pct", "wastage_cost"]]

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(monthly["month"], monthly["promo_revenue_share"], color=BLUE, linewidth=2, marker="o", markersize=5)
ax.set_title("Share of revenue from promotion orders, by month")
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_ylim(0, None)
fig.tight_layout()
fig.savefig(FIG_DIR / "promo_revenue_share_by_month.png", dpi=130)
plt.close(fig)
FIGS["promo_monthly"] = "figures/promo_revenue_share_by_month.png"

promo_split = (fact_m.groupBy("is_promo").agg(
                   F.sum("line_total").alias("revenue"), F.sum("line_cost").alias("cost"),
                   F.countDistinct("order_id").alias("orders"),
                   F.sum("line_discount_amount").alias("discount"))
               .toPandas())
promo_split["revenue_share"] = promo_split["revenue"] / promo_split["revenue"].sum()
promo_split["margin_pct"] = (promo_split["revenue"] - promo_split["cost"]) / promo_split["revenue"] * 100
promo_split["avg_order_value"] = promo_split["revenue"] / promo_split["orders"]
promo_split["order_type"] = promo_split["is_promo"].map({True: "Promotion order", False: "No promotion"})
promo_split = promo_split.sort_values("is_promo", ascending=False)
R["promo_split"] = promo_split[["order_type", "orders", "revenue", "revenue_share", "avg_order_value", "margin_pct"]]
PROMO_SHARE = promo_split.loc[promo_split["is_promo"], "revenue_share"].iloc[0]

# A location-day is "in a promotion period" when any promotion running at that location
# covers that date, whether or not a given order used it.
promos = load_clean_table(spark, "Promotions", PROCESSED_DIR)
promo_locs = load_clean_table(spark, "Promotion_Locations", PROCESSED_DIR)
promo_days = (promos.join(promo_locs, "promotion_id")
              .select("location_id", F.explode(F.sequence("start_date", "end_date")).alias("day"))
              .distinct().withColumn("in_promo_period", F.lit(True)))
loc_day = (fact.groupBy("location_id", F.to_date("order_datetime").alias("day"))
           .agg(F.sum("line_total").alias("revenue"), F.countDistinct("order_id").alias("orders"))
           .join(promo_days, ["location_id", "day"], "left")
           .fillna(False, subset=["in_promo_period"]))
period = (loc_day.groupBy("in_promo_period").agg(
              F.count("*").alias("location_days"),
              F.sum("revenue").alias("revenue"),
              F.avg("revenue").alias("avg_daily_revenue"),
              F.avg("orders").alias("avg_daily_orders"))
          .toPandas().sort_values("in_promo_period", ascending=False))
period["revenue_share"] = period["revenue"] / period["revenue"].sum()
period["period"] = period["in_promo_period"].map({True: "During a promotion period", False: "Outside promotion periods"})
R["promo_period"] = period[["period", "location_days", "revenue_share", "avg_daily_revenue", "avg_daily_orders"]]
PERIOD_LIFT = period.set_index("in_promo_period").loc[True, "avg_daily_revenue"] / period.set_index("in_promo_period").loc[False, "avg_daily_revenue"] - 1

by_promo = (fact.filter(F.col("promotion_id").isNotNull()).groupBy("promotion_id").agg(
                F.sum("line_total").alias("revenue"), F.sum("line_cost").alias("cost"),
                F.countDistinct("order_id").alias("orders"))
            .join(promos.select("promotion_id", "promotion_name", "promotion_type"), "promotion_id")
            .toPandas())
by_promo["margin_pct"] = (by_promo["revenue"] - by_promo["cost"]) / by_promo["revenue"] * 100
by_promo = by_promo.sort_values("revenue", ascending=False)
R["by_promo"] = by_promo[["promotion_name", "promotion_type", "orders", "revenue", "margin_pct"]]

jul, aug = monthly.set_index("order_month").loc[7], monthly.set_index("order_month").loc[8]
AUG = dict(orders=aug["orders"] / jul["orders"] - 1, customers=aug["customers"] / jul["customers"] - 1,
           revenue=aug["revenue"] / jul["revenue"] - 1, margin_jul=jul["margin_pct"], margin_aug=aug["margin_pct"],
           waste_x=aug["wastage_cost"] / jul["wastage_cost"])
print(AUG, f"promo share {PROMO_SHARE:.1%}, period lift {PERIOD_LIFT:.1%}")
display(R["monthly"]); display(R["promo_split"]); display(R["promo_period"]); display(R["by_promo"])

/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


{'orders': np.float64(0.5753351614868982), 'customers': np.float64(0.3750503018108653), 'revenue': np.float64(0.6231614249047186), 'margin_jul': np.float64(57.43582418322407), 'margin_aug': np.float64(53.91452952378047), 'waste_x': np.float64(3.5511496753838645)} promo share 18.8%, period lift 29.8%


,month,orders,customers,revenue,promo_revenue_share,margin_pct,wastage_cost
0,Jan,14675,10829,9.088071e+07,0.177494,57.239719,1950967.54
1,Feb,13021,9920,8.358010e+07,0.095694,57.600627,1626687.10
2,Mar,13512,10272,9.398882e+07,0.251754,56.852845,2334780.27
3,Apr,14878,11008,1.003553e+08,0.134681,55.351457,1865130.37
4,May,14840,10921,9.432813e+07,0.197608,56.763592,1948363.34
5,Jun,13109,9910,8.519376e+07,0.206405,56.916178,1733612.69
6,Jul,13128,9940,8.531552e+07,0.204201,57.435824,1858133.20
7,Aug,20681,13668,1.384809e+08,0.254519,53.914530,6598509.11
8,Sep,11923,9043,8.146697e+07,0.178075,56.491890,1874497.81
9,Oct,12756,9650,8.907322e+07,0.106976,56.601425,1922289.98


,order_type,orders,revenue,revenue_share,avg_order_value,margin_pct
0,Promotion order,31979,2.261722e+08,0.188409,7072.523213,51.428679
1,No promotion,147304,9.742581e+08,0.811591,6613.928644,57.623897


,period,location_days,revenue_share,avg_daily_revenue,avg_daily_orders
0,During a promotion period,7888,0.892192,135777.710423,20.314275
1,Outside promotion periods,1237,0.107808,104620.683104,15.395311


,promotion_name,promotion_type,orders,revenue,margin_pct
12,August BOGO Blast,BOGO,4660,33294250.00,42.339719
19,Summer Coolers,PercentOff,5140,29681948.75,55.574190
6,Ramadan Iftar Deal,FixedAmountOff,2512,22860041.64,55.919208
15,Winter Soup Season,PercentOff,3155,21657680.75,57.453326
13,Monsoon Munchies,FixedAmountOff,2664,16622671.33,56.725609
5,Family Platter Month,ComboDeal,1983,16352351.00,47.915751
9,Winter Warmers,PercentOff,2607,15929703.00,56.914156
14,App Exclusive Fest,PercentOff,1811,11961780.00,45.850033
3,Spring Pizza Fest,BOGO,1107,9341650.00,40.153080
7,Year End Celebration,FixedAmountOff,960,8824591.85,53.972914


## Customer overview

In [12]:
cust_summary = customers[["customer_recency", "customer_frequency", "customer_monetary_value",
                          "average_order_value", "basket_size_avg", "peak_hour_frequency",
                          "weekend_order_ratio"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T
cust_summary = cust_summary[["mean", "25%", "50%", "75%", "90%", "99%"]].reset_index().rename(columns={"index": "feature"})
R["customers"] = cust_summary
ONE_TIME = (customers["customer_frequency"] == 1).mean()
top10 = customers.nlargest(int(len(customers) * 0.1), "customer_monetary_value")["customer_monetary_value"].sum()
TOP10_SHARE = top10 / customers["customer_monetary_value"].sum()
print(f"one-time customers {ONE_TIME:.1%}; top 10% of customers = {TOP10_SHARE:.1%} of spend")
display(cust_summary)

one-time customers 43.4%; top 10% of customers = 43.6% of spend


,feature,mean,25%,50%,75%,90%,99%
0,customer_recency,113.748115,22.00,80.000000,191.000000,277.000000,352.0000
1,customer_frequency,3.614966,1.00,2.000000,4.000000,9.000000,20.0000
2,customer_monetary_value,28393.616354,6842.50,14133.150000,32958.150000,67951.725000,191348.3750
3,average_order_value,7428.898716,5083.00,6998.734762,9214.452917,11747.160417,18749.4275
4,basket_size_avg,5.250365,4.00,5.000000,6.000000,7.000000,10.0000
5,peak_hour_frequency,0.560995,0.25,0.571429,1.000000,1.000000,1.0000
6,weekend_order_ratio,0.517993,0.00,0.500000,1.000000,1.000000,1.0000


## Write the report

In [13]:
def t(key, cols, formats=None):
    df = R[key][list(cols)].rename(columns=cols)
    fm = {cols[k]: v for k, v in (formats or {}).items()}
    return md_table(df, fm)


def fig(key, alt):
    return f"![{alt}]({FIGS[key]})"


ITEM_COLS = {"item_name": "Dish", "category_name": "Category"}
money_m = lambda v: f"{v / 1e6:,.1f}M"
first = lambda df, col="item_name", n=3: ", ".join(df[col].head(n))

lm_pop = R["lowest_margin"][R["lowest_margin"]["item_popularity"] >= 0.85]
lm_range = f'{lm_pop["profit_percentage"].min():.0f}-{lm_pop["profit_percentage"].max():.0f}%'
promo_waste = R["high_wastage"][R["high_wastage"]["promotion_dependency"] >= 0.5]
volume_waste = R["high_wastage_cost"].head(4)
weak_promos = R["by_promo"].nsmallest(3, "margin_pct")
loc_m = R["locations"]["profit_margin_pct"]
under50 = R["category_revenue"][R["category_revenue"]["margin_pct"] < 50]["category_name"].tolist()
high_margin_low_vol = int((R["highest_margin"]["item_popularity"] <= 0.5).sum())
poor = R["worst_rated"]
poor_cut = poor[poor["average_rating"] < 2.5]
dt_row = R["location_types"].set_index("location_type")
ch_row = R["channels"].set_index("order_channel")
monthly_idx = R["monthly"].set_index("month")
promo_row = R["promo_split"].set_index("order_type")
bogo = R["by_promo"].set_index("promotion_name").loc["August BOGO Blast"]
sat = R["dow"].set_index("day").loc["Sat", "avg_orders_per_day"]
tue = R["dow"].set_index("day").loc["Tue", "avg_orders_per_day"]
top_cat = R["category_revenue"].iloc[0]
top_waste_cat = R["category_revenue"].sort_values("wastage_cost", ascending=False).iloc[0]
cust = customers
biggest, smallest = R["locations"].iloc[0], R["locations"].iloc[-1]

report = f"""# DineIQ — Exploratory Data Analysis

Data: all orders from 2025-01-01 through {AS_OF} (feature snapshot `as_of_date={AS_OF}`),
{len(items)} menu items, {len(locations)} locations. Revenue, volume and margin exclude cancelled
orders. Amounts are in PKR. Peak hours are 12:00-14:00 and 19:00-22:00; the weekend is
Friday-Sunday. Source notebook: `notebooks/01_eda.ipynb`. Feature definitions:
`spark_jobs/04_feature_engineering.py`.

## Headline numbers

| Metric | Value |
|---|---|
| Revenue (net of discounts, excl. tax) | {TOTAL_REVENUE / 1e6:,.1f}M PKR |
| Gross margin | {TOTAL_MARGIN / TOTAL_REVENUE:.1%} |
| Completed orders | {N_ORDERS:,} |
| Ordering customers | {len(customers):,} |
| Wastage cost | {WASTE_TOTAL / 1e6:,.1f}M PKR ({WASTE_TOTAL / TOTAL_REVENUE:.1%} of revenue) |
| Average dish rating | {AVG_RATING:.2f} / 5 |

## Key findings

- **Volume and margin are separate things.** {len(lm_pop)} of the menu's lowest-margin dishes are
  also in the top 15% by volume ({first(lm_pop, n=4)} and others). They run at {lm_range} margin,
  against a menu median of {MEDIAN_MARGIN:.1f}%. The 10 best sellers average
  {TOP10_VOL_MARGIN:.1f}% margin. Across the menu, rank correlation between units sold and margin is
  {VOL_MARGIN_CORR:.2f}, i.e. none.
- **Wastage is concentrated.** Six dishes account for {WASTE_TOP6_SHARE:.0%} of all wastage cost,
  led by {first(R["high_wastage_cost"])}. The Karahi & Handi category alone wastes
  {top_waste_cat["wastage_cost"] / 1e6:,.1f}M PKR.
- **{len(poor_cut)} dishes are rated far below the rest** ({first(poor_cut, n=len(poor_cut))}), at
  {poor_cut["average_rating"].min():.2f}-{poor_cut["average_rating"].max():.2f} against a
  {AVG_RATING:.2f} average. The next-worst dish is at {poor[poor["average_rating"] >= 2.5]["average_rating"].min():.2f}.
- **Weekends carry the business.** Friday-Sunday average {WEEKEND_LIFT:.0%} more orders per day than
  Monday-Thursday. {PEAK_SHARE:.0%} of orders fall in the two peak windows, and 20:00-21:00 is the
  single busiest hour.
- **The August BOGO Blast grew volume but cost margin.** Against July, August orders rose
  {AUG["orders"]:.0%} and unique customers {AUG["customers"]:.0%}. Gross margin fell from
  {AUG["margin_jul"]:.1f}% to {AUG["margin_aug"]:.1f}%, and wastage cost was
  {AUG["waste_x"]:.1f}x July's. The BOGO orders themselves ran at {bogo["margin_pct"]:.1f}% margin.
- **Promotions lift traffic at lower margin.** Promotion orders are {PROMO_SHARE:.1%} of revenue at
  {promo_row.loc["Promotion order", "margin_pct"]:.1f}% margin, versus
  {promo_row.loc["No promotion", "margin_pct"]:.1f}% for other orders. Location-days inside a
  promotion period take {PERIOD_LIFT:.0%} more revenue than days outside one.
- **Dine-in is half of revenue** ({ch_row.loc["Dine-in", "revenue_share"]:.0%}), with the largest
  baskets and the lowest cancellation rate ({ch_row.loc["Dine-in", "cancel_rate"]:.1%}).
  Third-party delivery cancels {ch_row.loc["ThirdPartyDelivery", "cancel_rate"]:.1%} of orders.
- **Location revenue varies {LOC_SPREAD:.1f}x**, from {biggest["location_name"]}
  ({biggest["location_revenue"] / 1e6:,.1f}M) to {smallest["location_name"]}
  ({smallest["location_revenue"] / 1e6:,.1f}M). Margin is flat across locations ({loc_m.min():.0f}-{loc_m.max():.0f}%). The spread
  comes from volume, not pricing.

## 1. Dish rankings

### 1.1 Top-selling dishes

The best sellers are cheap staples and add-ons: breads, fries and soft drinks. Mains take over when
ranked by revenue (1.3).

{fig("top_selling", "Top-selling dishes")}

{t("top_selling", {**ITEM_COLS, "total_quantity_sold": "Units sold", "item_revenue": "Revenue (PKR)",
                    "profit_percentage": "Margin", "average_rating": "Avg rating"},
   {"total_quantity_sold": intf, "item_revenue": pkr, "profit_percentage": pct, "average_rating": num2})}

### 1.2 Lowest-selling dishes

{NEW_IN_BOTTOM} of the {TOP_N} lowest-selling dishes were launched during the year, mostly in Q4.
Their totals reflect limited time on the menu, not weak demand. The per-day rate is the fairer
comparison.

{t("lowest_selling", {**ITEM_COLS, "total_quantity_sold": "Units sold", "units_per_day": "Units/day on menu",
                       "launch_date": "Launched", "item_revenue": "Revenue (PKR)"},
   {"total_quantity_sold": intf, "units_per_day": lambda v: f"{v:.1f}", "item_revenue": pkr})}

Slowest sellers among dishes on the menu for at least 180 days, by units per day:

{t("lowest_rate", {**ITEM_COLS, "units_per_day": "Units/day", "total_quantity_sold": "Units sold",
                    "profit_percentage": "Margin", "average_rating": "Avg rating"},
   {"units_per_day": lambda v: f"{v:.1f}", "total_quantity_sold": intf, "profit_percentage": pct,
    "average_rating": num2})}

### 1.3 Highest-revenue dishes

Karahi and handi dishes lead revenue. {R["highest_revenue"].iloc[0]["item_name"]} is #1 despite
selling fewer units than {R["highest_revenue"].iloc[1]["item_name"]}.

{fig("highest_revenue", "Highest-revenue dishes")}

{t("highest_revenue", {**ITEM_COLS, "item_revenue": "Revenue (PKR)", "total_quantity_sold": "Units sold",
                        "profit_percentage": "Margin"},
   {"item_revenue": pkr, "total_quantity_sold": intf, "profit_percentage": pct})}

### 1.4 Highest-profit dishes (contribution margin in PKR)

{fig("highest_profit", "Highest-profit dishes")}

{t("highest_profit", {**ITEM_COLS, "contribution_margin": "Contribution margin (PKR)", "item_revenue": "Revenue (PKR)",
                       "profit_percentage": "Margin", "total_quantity_sold": "Units sold"},
   {"contribution_margin": pkr, "item_revenue": pkr, "profit_percentage": pct, "total_quantity_sold": intf})}

### 1.5 Highest-margin and lowest-margin dishes

The highest-margin dishes are mostly beverages and desserts. {high_margin_low_vol} of the
{TOP_N} sit in the bottom half by volume.

{t("highest_margin", {**ITEM_COLS, "profit_percentage": "Margin", "contribution_margin": "Contribution margin (PKR)",
                       "total_quantity_sold": "Units sold", "item_popularity": "Popularity pct."},
   {"profit_percentage": pct, "contribution_margin": pkr, "total_quantity_sold": intf, "item_popularity": num2})}

The lowest-margin list is the mirror image. Most of it is the menu's highest-volume staples, followed
by large combo/platter items that lean on promotions.

{t("lowest_margin", {**ITEM_COLS, "profit_percentage": "Margin", "total_quantity_sold": "Units sold",
                      "item_popularity": "Popularity pct."},
   {"profit_percentage": pct, "total_quantity_sold": intf, "item_popularity": num2})}

{fig("volume_vs_margin", "Units sold vs profit margin")}

### 1.6 High-wastage dishes

Wastage rate = wasted portions / (wasted + sold portions). The waste log records kg, liters or pieces.
Wasted quantity is converted to portions through its cost (waste cost / average unit cost). Two
groups stand out:

- promotion-dependent, low-volume dishes ({first(promo_waste, n=len(promo_waste))}; half or more
  of their revenue comes from promotion orders), where kitchens prepare for promotional demand that
  often does not arrive;
- high-volume curries and grills ({first(volume_waste, n=4)}), at
  {volume_waste["wastage_percentage"].min():.0%}-{volume_waste["wastage_percentage"].max():.0%} wasted, which
  make up most of the waste in PKR.

{fig("high_wastage", "High-wastage dishes")}

{t("high_wastage", {**ITEM_COLS, "wastage_percentage": "Wastage rate", "wastage_cost": "Wastage cost (PKR)",
                     "total_quantity_sold": "Units sold", "promotion_dependency": "Promo revenue share"},
   {"wastage_percentage": frac_pct, "wastage_cost": pkr, "total_quantity_sold": intf, "promotion_dependency": frac_pct})}

Ranked by wastage cost:

{t("high_wastage_cost", {**ITEM_COLS, "wastage_cost": "Wastage cost (PKR)", "wastage_percentage": "Wastage rate",
                          "total_quantity_sold": "Units sold"},
   {"wastage_cost": pkr, "wastage_percentage": frac_pct, "total_quantity_sold": intf})}

### 1.7 Best-rated dishes

Dishes with at least {MIN_RATINGS} ratings. Rating trend is the slope of monthly average rating,
in points per month.

{t("best_rated", {**ITEM_COLS, "average_rating": "Avg rating", "n_ratings": "Ratings",
                   "rating_trend": "Trend / month", "total_quantity_sold": "Units sold"},
   {"average_rating": num2, "n_ratings": intf, "rating_trend": lambda v: f"{v:+.3f}", "total_quantity_sold": intf})}

### 1.8 Poorly rated dishes

The bottom {len(poor_cut)} are a distinct group, well below everything else, and their trends are
flat: they are not recovering. They still sell ({poor_cut["total_quantity_sold"].sum():,} units
combined), so they are candidates for recipe review rather than removal on volume grounds alone.

{fig("worst_rated", "Poorly rated dishes")}

{t("worst_rated", {**ITEM_COLS, "average_rating": "Avg rating", "n_ratings": "Ratings", "rating_trend": "Trend / month",
                    "total_quantity_sold": "Units sold", "repeat_purchase_rate": "Repeat rate"},
   {"average_rating": num2, "n_ratings": intf, "rating_trend": lambda v: f"{v:+.3f}", "total_quantity_sold": intf,
    "repeat_purchase_rate": frac_pct})}

## 2. Menu categories

{top_cat["category_name"]} is the largest category by revenue
({top_cat["revenue_share"]:.1%}). Order reach tells a different story: Sides & Breads and Cold
Beverages appear in over half of all orders but earn a fraction of the revenue. The categories under 50% margin are {", ".join(under50)}.

{fig("category_revenue", "Revenue by category")}

{t("category_revenue", {"category_name": "Category", "revenue": "Revenue (PKR)", "revenue_share": "Share",
                         "margin_pct": "Margin", "units": "Units", "wastage_cost": "Wastage cost (PKR)"},
   {"revenue": pkr, "revenue_share": frac_pct, "margin_pct": pct, "units": intf, "wastage_cost": pkr})}

{fig("category_orders", "Share of orders containing each category")}

{t("category_orders", {"category_name": "Category", "orders": "Orders containing", "order_penetration": "Share of orders",
                        "units": "Units"},
   {"orders": intf, "order_penetration": frac_pct, "units": intf})}

## 3. Peak ordering periods

Two daily peaks: lunch (12:00-14:00) and a larger dinner peak (19:00-22:00), together
{PEAK_SHARE:.0%} of orders. The busiest hours are
{", ".join(f"{h:02d}:00 ({v:.1%})" for h, v in TOP_HOURS.items())}. Saturday is the busiest day
({sat:,.0f} orders/day against {tue:,.0f} on Tuesday).

Ramadan (March) reshapes the day. Lunch drops to {RAMADAN_LUNCH:.1%} of orders (vs
{OTHER_LUNCH:.1%} in other months), and the peak moves to the iftar hour, {RAMADAN_PEAK_HOUR:02d}:00.
Staffing and prep schedules built on the normal lunch peak do not apply in that month.

{fig("heatmap", "Orders by day of week and hour")}

{t("dow", {"day": "Day", "avg_orders_per_day": "Avg orders/day", "share_of_week": "Share of week"},
   {"avg_orders_per_day": lambda v: f"{v:,.0f}", "share_of_week": frac_pct})}

## 4. Location-wise sales patterns

Mall locations earn the most per site, and highway sites the least, though highway sites have the
highest average order value ({dt_row.loc["Highway", "avg_order_value"]:,.0f} PKR). Highway sites
also have the highest wastage rate ({dt_row.loc["Highway", "wastage_rate"]:.1%} of revenue) and the
lowest repeat-customer rate, consistent with transient traffic. Downtown sites are the least
weekend-dependent ({dt_row.loc["Downtown", "weekend_share"]:.0%} of orders on Fri-Sun vs
{dt_row.loc["Mall", "weekend_share"]:.0%} for malls) and have the strongest lunch trade
({dt_row.loc["Downtown", "lunch_share"]:.0%}).

{t("location_types", {"location_type": "Type", "n_locations": "Sites", "revenue_per_location": "Revenue / site (PKR)",
                       "avg_order_value": "AOV (PKR)", "weekend_share": "Weekend share", "lunch_share": "Lunch share",
                       "dinner_share": "Dinner share", "repeat_rate": "Repeat-customer rate", "wastage_rate": "Wastage / revenue"},
   {"revenue_per_location": pkr, "avg_order_value": pkr, "weekend_share": frac_pct, "lunch_share": frac_pct,
    "dinner_share": frac_pct, "repeat_rate": frac_pct, "wastage_rate": frac_pct})}

{fig("locations", "Revenue by location")}

{t("locations", {"location_name": "Location", "city": "City", "location_type": "Type", "location_revenue": "Revenue (PKR)",
                  "profit_margin_pct": "Margin", "location_avg_order_value": "AOV (PKR)",
                  "location_customer_count": "Customers", "location_repeat_customer_rate": "Repeat rate",
                  "location_wastage_rate": "Wastage / revenue"},
   {"location_revenue": pkr, "profit_margin_pct": pct, "location_avg_order_value": pkr,
    "location_customer_count": intf, "location_repeat_customer_rate": frac_pct, "location_wastage_rate": frac_pct})}

## 5. Channel-wise ordering patterns

Dine-in orders are the largest ({ch_row.loc["Dine-in", "avg_order_value"]:,.0f} PKR,
{ch_row.loc["Dine-in", "avg_basket_size"]:.1f} lines on average). Delivery and takeaway orders are
around {ch_row.loc["Takeaway", "avg_order_value"]:,.0f}-{ch_row.loc["App", "avg_order_value"]:,.0f} PKR.
Margin is essentially identical across channels. The channel difference is in order size and
cancellation: third-party delivery cancels {ch_row.loc["ThirdPartyDelivery", "cancel_rate"] / ch_row.loc["Dine-in", "cancel_rate"]:.1f}x
as often as dine-in. Timing does not differ by channel; every channel does about
{ch_row["peak_share"].mean():.0%} of its orders in peak hours.

{fig("channels", "Revenue share by channel")}

{t("channels", {"order_channel": "Channel", "orders": "Orders", "revenue": "Revenue (PKR)", "revenue_share": "Share",
                 "avg_order_value": "AOV (PKR)", "avg_basket_size": "Lines/order", "margin_pct": "Margin",
                 "cancel_rate": "Cancel rate", "promo_order_share": "Promo orders", "peak_share": "Peak-hour share"},
   {"orders": intf, "revenue": pkr, "revenue_share": frac_pct, "avg_order_value": pkr,
    "avg_basket_size": lambda v: f"{v:.1f}", "margin_pct": pct, "cancel_rate": frac_pct,
    "promo_order_share": frac_pct, "peak_share": frac_pct})}

Customers' most-used channel:

{t("channel_pref", {"channel_preference": "Preferred channel", "share_of_customers": "Share of customers"},
   {"share_of_customers": frac_pct})}

## 6. Promotion-driven sales

Two views. First, orders that used a promotion against those that did not:

{t("promo_split", {"order_type": "Order type", "orders": "Orders", "revenue": "Revenue (PKR)",
                    "revenue_share": "Revenue share", "avg_order_value": "AOV (PKR)", "margin_pct": "Margin"},
   {"orders": intf, "revenue": pkr, "revenue_share": frac_pct, "avg_order_value": pkr, "margin_pct": pct})}

Second, revenue by location-day, depending on whether a promotion was running at that location on that
day. Promotions run on most days of the year, so "outside" is the smaller group:

{t("promo_period", {"period": "Period", "location_days": "Location-days", "revenue_share": "Revenue share",
                     "avg_daily_revenue": "Avg revenue / location-day (PKR)", "avg_daily_orders": "Avg orders / location-day"},
   {"location_days": intf, "revenue_share": frac_pct, "avg_daily_revenue": pkr, "avg_daily_orders": lambda v: f"{v:.1f}"})}

Promotion orders carry larger baskets (higher AOV) but {promo_row.loc["No promotion", "margin_pct"] - promo_row.loc["Promotion order", "margin_pct"]:.1f}
points less margin. The weakest campaigns on margin are
{", ".join(f"{n} ({m:.0f}%)" for n, m in zip(weak_promos["promotion_name"], weak_promos["margin_pct"]))},
all BOGO or deep percentage-off.

The August BOGO Blast is the clearest case. August was the second-highest revenue month
({monthly_idx.loc["Aug", "revenue"] / 1e6:,.1f}M), but it had the year's lowest gross margin
({AUG["margin_aug"]:.1f}%) and by far the highest wastage cost
({monthly_idx.loc["Aug", "wastage_cost"] / 1e6:,.1f}M PKR, {AUG["waste_x"]:.1f}x July). The waste spike
is consistent with kitchens over-preparing for the campaign. The volume gain came with margin and waste costs that the revenue
line alone does not show.

{fig("promo_monthly", "Promotion revenue share by month")}

{t("monthly", {"month": "Month", "orders": "Orders", "customers": "Customers", "revenue": "Revenue (PKR)",
                "promo_revenue_share": "Promo revenue share", "margin_pct": "Margin", "wastage_cost": "Wastage cost (PKR)"},
   {"orders": intf, "customers": intf, "revenue": pkr, "promo_revenue_share": frac_pct, "margin_pct": pct,
    "wastage_cost": pkr})}

By campaign:

{t("by_promo", {"promotion_name": "Promotion", "promotion_type": "Type", "orders": "Orders", "revenue": "Revenue (PKR)",
                 "margin_pct": "Margin"},
   {"orders": intf, "revenue": pkr, "margin_pct": pct})}

## 7. Customer overview

{ONE_TIME:.0%} of ordering customers have ordered only once. The top 10% of customers by spend
account for {TOP10_SHARE:.0%} of customer spend. These customer features feed the segmentation step.

{t("customers", {"feature": "Feature", "mean": "Mean", "25%": "P25", "50%": "Median", "75%": "P75", "90%": "P90", "99%": "P99"},
   {c: (lambda v: f"{v:,.2f}") for c in ["mean", "25%", "50%", "75%", "90%", "99%"]})}

## Method notes

- Features are computed as of {AS_OF} and use only records dated on or before it. An earlier
  snapshot ({snapshots[0]}) exists to validate the as-of logic. Later forecasting steps recompute
  features at their own cutoffs.
- Cancelled orders are excluded from revenue, volume, margin and customer metrics. They are kept only
  for the channel cancellation rate.
- Contribution margin = line revenue (after line discounts) - quantity x unit cost. Tax is excluded.
- Peak hours: 12:00-13:59 and 19:00-21:59. 14:00 and 22:00 are busy shoulder hours but fall outside
  the windows.
- `price_change_percentage` uses chain-wide Pricing_History rows only. Location premium rows are
  excluded.
"""
REPORT.write_text(report)
print(f"wrote {REPORT} ({len(report):,} chars)")

wrote /home/muzammil/Desktop/Techwiz 7/Data science/DineIQ/reports/eda_report.md (30,743 chars)
